# Chapter - 2: Coding attention mechanism

## 2.1 Attending to different parts of the i/p with self attention

### 2.2.1 A simple self-attention mechanism without trainabe weights

In [4]:
#Simple EG:
import torch

inputs = torch.tensor(
    [[0.43,0.15,0.89],# Your  (x^1)
     [0.55,0.87,0.66],# journey (x^2)
     [0.57,0.85,0.64],# starts (x^3)
     [0.22,0.58,0.33],# with (x^4)
     [0.77,0.25,0.10],# one (x^5)
     [0.05,0.80,0.55] # step (x^6)
    ]
)

In [5]:
input_query = inputs[1]                       #2nd vector in inputs
input_query

tensor([0.5500, 0.8700, 0.6600])

In [6]:
inp_1 = inputs[0]     #1st vector in inputs
inp_1

tensor([0.4300, 0.1500, 0.8900])

In [7]:
0.55*0.43 + 0.87*0.15 + 0.66*0.89           #dot product of input_query and inp_1

0.9544

In [8]:
torch.dot(input_query,inp_1)                #using pyTorch

tensor(0.9544)

In [9]:
res = 0.

for idx , ele in enumerate(inputs[0]):
    res += inputs[0][idx] * input_query[idx]

res

tensor(0.9544)

In [10]:
i = 0
res = torch.dot(inputs[i],input_query)
res

tensor(0.9544)

In [11]:
torch.empty(inputs.shape[0])

tensor([-2.1095e+34,  1.9912e-42,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00])

In [12]:
query = inputs[1]           #2nd i/p token is the query

attn_score_2 = torch.empty(inputs.shape[0])     # we are creating a empty tensor (vector) with 0s with the size of  the input vectors, i.e here 6, for efficient memory management     

for i, x_i in enumerate(inputs):
    attn_score_2[i] = torch.dot(x_i,query)  # dot product 

print(attn_score_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [13]:
# Normalizing each tensors  above
attn_weights_2_tmp = attn_score_2/attn_score_2.sum()
attn_weights_2_tmp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [14]:
# we are normalizing to make the summ of those vctors to be equal to 1
attn_weights_2_tmp.sum()

tensor(1.0000)

In [15]:
#Another way of normalization is by using sofmax
def softmax_native(x):                              #self implemented softmax
    return torch.exp(x)/torch.exp(x).sum()
softmax_native(attn_score_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [16]:
# In practical use, the torch version of softmax normalization, in some cases, the native defined f() might fail. 
attn_weights_2 = torch.softmax(attn_score_2,dim=0)
attn_weights_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [17]:
query = inputs[1]

context_vect_2 = torch.zeros(query.shape)

for i, x_i in enumerate(inputs):
    
    print(f"{attn_weights_2[i]} -----> {inputs[i]}")                       #just a visualization
    context_vect_2 += attn_weights_2[i] * inputs[i]

print(context_vect_2)

0.13854756951332092 -----> tensor([0.4300, 0.1500, 0.8900])
0.2378913015127182 -----> tensor([0.5500, 0.8700, 0.6600])
0.23327402770519257 -----> tensor([0.5700, 0.8500, 0.6400])
0.12399158626794815 -----> tensor([0.2200, 0.5800, 0.3300])
0.10818186402320862 -----> tensor([0.7700, 0.2500, 0.1000])
0.15811361372470856 -----> tensor([0.0500, 0.8000, 0.5500])
tensor([0.4419, 0.6515, 0.5683])


# 2.2.2 A simple self-attention mechanism without trainable weights

In [19]:
attn_score = torch.empty(6,6)                   

for i, x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_score[i,j] = torch.dot(x_i,x_j)  

print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [20]:
# In PyTorch, there is an efficient way of doing matrix multiplication, and that is as follows:

attn_score = inputs @ inputs.T
attn_score

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [21]:
# Normalizing the attn_score by softmax:

attn_weight = torch.softmax(attn_score,dim = 1)              #what the dim = 1 ----> means each of the rown sum equals to 1 (normalize)
attn_weight

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [22]:
# dim means:

attn_weight.sum(dim = 1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [23]:
# finding context vector for all the attention weights:

all_context_vector = attn_weight @ inputs
all_context_vector

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [24]:
# cleaned version of everything done in this session:

attn_score = inputs @ inputs.T
attn_weight = torch.softmax(attn_score,dim = 1) 

all_context_vector = attn_weight @ inputs
all_context_vector


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

# Implemting self-attention with trainable weights

## computing attention weights step-by-step

In [27]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [28]:
x_2 = inputs[1]
d_in = inputs.shape[1]              # dimension of the input <---- here its three(3)
d_out = 2                           #dimension of output    (output embedding size)

In [37]:
torch.manual_seed(123)

W_query =torch.nn.Parameter(torch.rand(d_in, d_out))                # torch.Parameter is like a wrapper around ot to make it trainable   
W_key =torch.nn.Parameter(torch.rand(d_in, d_out))
W_value =torch.nn.Parameter(torch.rand(d_in, d_out))

In [41]:
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)

In [45]:
W_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)

In [47]:
W_value

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)

In [51]:
query_2 = x_2 @ W_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [55]:
keys = inputs @ W_key
value = inputs @ W_value

In [57]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [59]:
value

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], grad_fn=<MmBackward0>)

In [61]:
key_2 = keys[1]
attn_scor22 = torch.dot(key_2,query_2) 
attn_scor22

tensor(1.8524, grad_fn=<DotBackward0>)

In [63]:
attn_score_2 = query_2 @ keys.T
attn_score_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [65]:
dim_k = keys.shape[1]

attn_weights_2 = torch.softmax(attn_score_2 / dim_k ** 0.5 , dim = -1)                        # dim_k ** 0.5 ----> sqrt(dim_k)

attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [67]:
context_vect_2 = attn_weights_2 @ value
context_vect_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)